<a href="https://colab.research.google.com/github/SedexDevVE/colab-inventory-manager/blob/main/explorador_archivo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
import os
import time
import unicodedata
from datetime import datetime
import ipywidgets as w
from IPython.display import display, clear_output, HTML

DIRECTORIO = '/content/drive/My Drive/InventarioColab'
# Cambia '/content/drive/My Drive/InventarioColab' por la ruta exacta de la
# carpeta en tu propio Google Drive.
# Ejemplo: DIRECTORIO = '/content/drive/My Drive/MiCarpeta/MisDocumentos'
EXTENSIONES_PERMITIDAS = ('.json', '.csv', '.pdf', '.txt')
FORMATO_FECHA = '%Y-%m-%d %H:%M:%S'
UMBRAL_INSERTION = 16

CLAVES_ORDEN = {
    'Nombre': 'nombre_norm',
    'Tamaño (KB)': 'tamano_kb',
    'Fecha de modificación': 'ultima_modificacion',
}

def montar_drive(punto_montaje='/content/drive'):
    try:
        from google.colab import drive
    except ImportError:
        return False
    drive.mount(punto_montaje)
    return True

def normalizar(texto):
    texto = unicodedata.normalize('NFD', str(texto).lower())
    return ''.join(c for c in texto if unicodedata.category(c) != 'Mn')

def listar_archivos(directorio=DIRECTORIO, extensiones=EXTENSIONES_PERMITIDAS):
    try:
        nombres = os.listdir(directorio)
    except FileNotFoundError:
        return [], 'Directorio no encontrado.'
    except NotADirectoryError:
        return [], 'La ruta indicada no es un directorio.'
    except PermissionError:
        return [], 'Sin permisos para leer el directorio.'

    extensiones = tuple(e.lower() for e in extensiones)
    archivos = []
    for nombre in nombres:
        ruta = os.path.join(directorio, nombre)
        ext = os.path.splitext(nombre)[1].lower()
        if not os.path.isfile(ruta) or ext not in extensiones:
            continue
        archivos.append({
            'nombre': nombre,
            'nombre_norm': normalizar(nombre),
            'extension': ext,
            'tamano_kb': round(os.path.getsize(ruta) / 1024, 2),
            'ultima_modificacion': datetime.fromtimestamp(
                os.path.getmtime(ruta)).strftime(FORMATO_FECHA),
        })
    return archivos, None

# ----------------------------- Ordenamiento ---------------------------
def _va_antes(a, b, descendente):
    return a >= b if descendente else a <= b

def insertion_sort(lista, clave, descendente=False, stats=None):
    stats = stats if stats is not None else {'comparaciones': 0}
    res = list(lista)
    for i in range(1, len(res)):
        actual = res[i]
        k = clave(actual)
        j = i - 1
        while j >= 0:
            stats['comparaciones'] += 1
            if _va_antes(clave(res[j]), k, descendente):
                break
            res[j + 1] = res[j]
            j -= 1
        res[j + 1] = actual
    return res

def merge_sort(lista, clave, descendente=False, stats=None):
    stats = stats if stats is not None else {'comparaciones': 0}
    if len(lista) <= 1:
        return list(lista)
    medio = len(lista) // 2
    izq = merge_sort(lista[:medio], clave, descendente, stats)
    der = merge_sort(lista[medio:], clave, descendente, stats)
    res, i, j = [], 0, 0
    while i < len(izq) and j < len(der):
        stats['comparaciones'] += 1
        if _va_antes(clave(izq[i]), clave(der[j]), descendente):
            res.append(izq[i]); i += 1
        else:
            res.append(der[j]); j += 1
    res.extend(izq[i:])
    res.extend(der[j:])
    return res

def ordenar(lista, clave, descendente=False):
    stats = {'comparaciones': 0}
    if len(lista) <= UMBRAL_INSERTION:
        return insertion_sort(lista, clave, descendente, stats), 'Insertion Sort', stats['comparaciones']
    return merge_sort(lista, clave, descendente, stats), 'Merge Sort', stats['comparaciones']

# ----------------------------- Búsqueda -------------------------------
def buscar(archivos, consulta='', extension='Todas', min_kb=None, max_kb=None,
           desde=None, hasta=None, orden_por='nombre_norm', descendente=False):
    inicio = time.perf_counter()
    palabras = normalizar(consulta).split()

    def cumple(a):
        if palabras and not all(p in a['nombre_norm'] for p in palabras):
            return False
        if extension != 'Todas' and a['extension'] != extension:
            return False
        if min_kb is not None and a['tamano_kb'] < min_kb:
            return False
        if max_kb is not None and a['tamano_kb'] > max_kb:
            return False
        dia = a['ultima_modificacion'][:10]
        if desde and dia < desde:
            return False
        if hasta and dia > hasta:
            return False
        return True

    filtrados = [a for a in archivos if cumple(a)]
    ordenados, algoritmo, comps = ordenar(filtrados, lambda a: a[orden_por], descendente)
    tiempo_ms = round((time.perf_counter() - inicio) * 1000, 3)

    return {
        'resultados': ordenados,
        'total': len(ordenados),
        'algoritmo': algoritmo,
        'comparaciones': comps,
        'tiempo_ms': tiempo_ms,
    }

# ----------------------------- HTML & UX -------------------------------
def formatear_html(r):
    if r['total'] == 0:
        return """
        <div style="padding: 40px 20px; background: rgba(239, 68, 68, 0.08); color: #f87171; border-radius: 16px; border: 1px solid rgba(239, 68, 68, 0.2); font-family: 'Inter', sans-serif; text-align: center; margin-top: 15px;">
            <div style="font-size: 32px; margin-bottom: 8px;">🔍</div>
            <h4 style="margin: 0 0 6px 0; font-size: 16px; font-weight: 600;">Sin resultados</h4>
            <p style="margin: 0; font-size: 13px; color: #94a3b8;">Intenta ajustar los parámetros de búsqueda o limpiar los filtros.</p>
        </div>
        """

    html = f"""
    <div style="font-family: 'Inter', sans-serif; margin-top: 15px;">
        <!-- Barra de métricas superiores tipo Dashboard -->
        <div style="display: flex; flex-wrap: wrap; gap: 12px; margin-bottom: 16px; background: #1f2533; padding: 14px 20px; border-radius: 14px; border: 1px solid #2d3548; font-size: 13px; color: #94a3b8; align-items: center;">
            <div style="display: flex; align-items: center; gap: 6px;">
                <span>⚙️ Algoritmo:</span> <span style="color: #60a5fa; font-family: monospace; font-weight: 600;">{r['algoritmo']}</span>
            </div>
            <div style="height: 14px; width: 1px; background: #334155;"></div>
            <div style="display: flex; align-items: center; gap: 6px;">
                <span>🔄 Comparaciones:</span> <span style="color: #60a5fa; font-family: monospace; font-weight: 600;">{r['comparaciones']}</span>
            </div>
            <div style="height: 14px; width: 1px; background: #334155;"></div>
            <div style="display: flex; align-items: center; gap: 6px;">
                <span>⏱️ Tiempo:</span> <span style="color: #60a5fa; font-family: monospace; font-weight: 600;">{r['tiempo_ms']} ms</span>
            </div>
            <div style="margin-left: auto; display: flex; align-items: center; gap: 8px;">
                <span>Archivos encontrados:</span>
                <span style="background: #1d9bf0; color: #ffffff; padding: 3px 10px; border-radius: 20px; font-weight: 700; font-size: 12px;">{r['total']}</span>
            </div>
        </div>

        <!-- Grid de Tarjetas de Archivos -->
        <div style="display: grid; grid-template-columns: repeat(auto-fill, minmax(240px, 1fr)); gap: 14px;">
    """

    for a in r['resultados']:
        ext = a['extension']
        badge_bg, badge_color, icono = "#2a3142", "#cbd5e1", "📄"
        if ext == '.json':
            badge_bg, badge_color, icono = "rgba(52, 211, 153, 0.15)", "#34d399", "📋"
        elif ext == '.csv':
            badge_bg, badge_color, icono = "rgba(56, 189, 248, 0.15)", "#38bdf8", "📊"
        elif ext == '.pdf':
            badge_bg, badge_color, icono = "rgba(248, 113, 113, 0.15)", "#f87171", "📕"
        elif ext == '.txt':
            badge_bg, badge_color, icono = "rgba(148, 163, 184, 0.15)", "#cbd5e1", "📝"

        html += f"""
        <div style="background: #1f2533; border: 1px solid #2d3548; border-radius: 16px; padding: 16px; box-shadow: 0 4px 12px rgba(0,0,0,0.15); display: flex; flex-direction: column; justify-content: space-between;">
            <div>
                <div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 10px;">
                    <span style="font-size: 20px;">{icono}</span>
                    <span style="background: {badge_bg}; color: {badge_color}; font-size: 10px; font-weight: 700; padding: 3px 8px; border-radius: 6px; letter-spacing: 0.5px; text-transform: uppercase;">{ext}</span>
                </div>
                <div style="font-weight: 600; color: #ffffff; font-size: 13px; margin-bottom: 10px; word-break: break-all; line-height: 1.4;">
                    {a['nombre']}
                </div>
            </div>
            <div style="border-top: 1px solid #2d3548; padding-top: 10px; font-size: 12px; color: #64748b; display: flex; justify-content: space-between; align-items: center;">
                <span style="color: #94a3b8;">💾 <b>{a['tamano_kb']}</b> KB</span>
                <span style="color: #64748b; font-size: 11px;">{a['ultima_modificacion']}</span>
            </div>
        </div>
        """

    html += "</div></div>"
    return html

# ----------------------------- Interfaz -------------------------------
def lanzar_interfaz(directorio=DIRECTORIO):
    archivos, error = listar_archivos(directorio)
    if error:
        print(f"❌ Error crítico: {error}")
        return

    # Inyección CSS Global con corrección de altura e interlineado para widgets
    display(HTML("""
    <style>
        @import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&display=swap');

        .widget-box, .widget-inline-flex {
            font-family: 'Inter', sans-serif !important;
        }
        .widget-text input, .widget-dropdown select, .widget-numeric-text input, .widget-date-picker input {
            background-color: #272d3d !important;
            color: #ffffff !important;
            border: 1px solid #374151 !important;
            border-radius: 10px !important;
            padding: 8px 12px !important;
            font-size: 13px !important;
            line-height: normal !important;
            height: auto !important;
            box-sizing: border-box !important;
        }
        .widget-dropdown select {
            padding-right: 28px !important;
        }
        .widget-text input:focus, .widget-dropdown select:focus {
            border-color: #1d9bf0 !important;
            outline: none !important;
        }
        .widget-toggle-buttons button {
            background-color: #272d3d !important;
            color: #94a3b8 !important;
            border: 1px solid #374151 !important;
            border-radius: 8px !important;
            font-size: 12px !important;
            font-weight: 500 !important;
        }
        .widget-toggle-buttons .mod-primary, .widget-toggle-buttons button.active {
            background-color: #1d9bf0 !important;
            color: #ffffff !important;
            border-color: #1d9bf0 !important;
        }
    </style>
    """))

    def crear_label(texto):
        return w.HTML(f'<span style="font-family: \'Inter\', sans-serif; font-size: 12px; font-weight: 500; color: #94a3b8; margin-bottom: 6px; display: block;">{texto}</span>')

    # Componentes del formulario
    txt = w.Text(placeholder='ej. reporte_final', layout=w.Layout(width='100%'))
    ext = w.Dropdown(options=['Todas'] + list(EXTENSIONES_PERMITIDAS), layout=w.Layout(width='100%'))

    mn = w.FloatText(value=0, placeholder='0', layout=w.Layout(width='100%'))
    mx = w.FloatText(value=0, placeholder='Max KB', layout=w.Layout(width='100%'))

    desde = w.DatePicker(layout=w.Layout(width='100%'))
    hasta = w.DatePicker(layout=w.Layout(width='100%'))

    orden = w.Dropdown(options=list(CLAVES_ORDEN), layout=w.Layout(width='100%'))
    sentido = w.ToggleButtons(options=['Ascendente', 'Descendente'], button_style='')

    boton = w.Button(
        description='FILTRAR RESULTADOS',
        button_style='',
        icon='search',
        layout=w.Layout(width='100%', height='42px', margin='16px 0 0 0')
    )
    boton.style.button_color = '#1d9bf0'

    salida = w.Output(layout=w.Layout(flex='1', width='100%'))

    def ejecutar(_=None):
        r = buscar(
            archivos, txt.value, ext.value,
            min_kb=mn.value if mn.value > 0 else None,
            max_kb=mx.value if mx.value > 0 else None,
            desde=desde.value.isoformat() if desde.value else None,
            hasta=hasta.value.isoformat() if hasta.value else None,
            orden_por=CLAVES_ORDEN[orden.value],
            descendente=(sentido.value == 'Descendente'),
        )
        with salida:
            clear_output()
            display(HTML(formatear_html(r)))

    boton.on_click(ejecutar)

    # Columna Izquierda: Hero Text y Resultados
    panel_izquierdo = w.VBox([
        w.HTML("""
        <div style="padding-right: 10px;">
            <span style="background: rgba(29, 155, 240, 0.15); color: #1d9bf0; font-size: 11px; font-weight: 700; padding: 4px 10px; border-radius: 20px; text-transform: uppercase; letter-spacing: 0.5px;">
                Explorador de Archivos
            </span>
            <h1 style="color: #ffffff; font-size: 28px; font-weight: 700; margin: 12px 0 8px 0; line-height: 1.25; letter-spacing: -0.5px;">
                Visualiza y Filtra tus Archivos en <span style="color: #1d9bf0;">GOOGLE DRIVE</span> con Facilidad.
            </h1>
            <p style="color: #94a3b8; font-size: 13px; line-height: 1.5; margin-bottom: 16px;">
                Aplica filtros avanzados en tiempo real, analiza comparaciones de ordenamiento y visualiza tus metadatos con rapidez.
            </p>
        </div>
        """),
        salida
    ], layout=w.Layout(flex='1.3', width='100%'))

    # Columna Derecha: Tarjeta de Filtros Avanzados
    panel_derecho = w.VBox([
        w.HTML("""
        <div style="margin-bottom: 14px;">
            <h2 style="margin: 0; font-size: 18px; color: #ffffff; font-weight: 700;">FILTROS AVANZADOS</h2>
            <p style="margin: 4px 0 0 0; font-size: 12px; color: #64748b;">Ajusta tus criterios</p>
        </div>
        """),
        crear_label("Búsqueda por nombre"), txt,
        w.Box(layout=w.Layout(height='8px')),

        crear_label("Extensión"), ext,
        w.Box(layout=w.Layout(height='8px')),

        crear_label("Tamaño (KB)"),
        w.HBox([mn, mx], layout=w.Layout(gap='8px', width='100%')),
        w.Box(layout=w.Layout(height='8px')),

        w.HBox([
            w.VBox([crear_label("Desde"), desde], layout=w.Layout(flex='1')),
            w.VBox([crear_label("Hasta"), hasta], layout=w.Layout(flex='1'))
        ], layout=w.Layout(gap='8px', width='100%')),
        w.Box(layout=w.Layout(height='8px')),

        crear_label("Ordenar por"), orden,
        w.Box(layout=w.Layout(height='8px')),

        crear_label("Dirección de orden"), sentido,

        boton
    ], layout=w.Layout(
        flex='0.9',
        backgroundColor='#1f2533',
        padding='20px',
        borderRadius='20px',
        border='1px solid #2d3548',
        boxShadow='0 20px 25px -5px rgba(0, 0, 0, 0.3)'
    ))

    # Layout Principal de 2 Columnas
    cuerpo = w.HBox([panel_izquierdo, panel_derecho], layout=w.Layout(gap='24px', width='100%'))

    # Contenedor Padre
    app = w.VBox([cuerpo], layout=w.Layout(
        backgroundColor='#141721',
        padding='16px 24px 24px 24px',
        borderRadius='20px',
        width='100%'
    ))

    display(app)
    ejecutar()

if __name__ == '__main__':
    montar_drive()
    lanzar_interfaz()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [11]:
import csv
import json
import os
import random
from datetime import datetime, timedelta

DIRECTORIO = '/content/drive/My Drive/InventarioColab'
# Cambia esta ruta si deseas utilizar otra carpeta en tu propio Google Drive.
# Ejemplo: DIRECTORIO = '/content/drive/My Drive/MiCarpetaPersonalizada/Inventario'

TEMAS = ['inventario', 'ventas', 'compras', 'clientes', 'proveedores',
         'productos', 'reporte', 'pedidos', 'facturas', 'almacén', 'devoluciones']
PERIODOS = ['enero', 'febrero', 'marzo', 'abril', 'mayo', 'junio',
            'julio', 'agosto', 'septiembre', 'octubre', '2026_Q1', '2026_Q2', 'final']
CATEGORIAS = ['Electrónica', 'Papelería', 'Limpieza', 'Alimentos', 'Herramientas']


def _registros(n):
    return [{'id': i + 1,
             'producto': f'Producto {i + 1}',
             'categoria': random.choice(CATEGORIAS),
             'cantidad': random.randint(0, 500),
             'precio': round(random.uniform(1, 999), 2)} for i in range(n)]


def _guardar_pdf(ruta, regs):
    """Genera un archivo PDF válido en puro Python sin dependencias externas."""
    lineas_texto = [f"ID {r['id']}: {r['producto']} | {r['categoria']} | Cant: {r['cantidad']} | ${r['precio']}" for r in regs[:15]]
    stream_content = "BT /F1 10 Tf 40 750 Td (" + ") Tj 0 -18 Td (".join(lineas_texto) + ") ET"
    stream_bytes = stream_content.encode('latin-1', errors='ignore')

    pdf_data = bytearray()
    pdf_data.extend(b"%PDF-1.4\n")

    obj1 = b"1 0 obj\n<< /Type /Catalog /Pages 2 0 R >>\nendobj\n"
    obj2 = b"2 0 obj\n<< /Type /Pages /Kids [3 0 R] /Count 1 >>\nendobj\n"
    obj3 = b"3 0 obj\n<< /Type /Page /Parent 2 0 R /Resources << /Font << /F1 4 0 R >> >> /MediaBox [0 0 612 792] /Contents 5 0 R >>\nendobj\n"
    obj4 = b"4 0 obj\n<< /Type /Font /Subtype /Type1 /BaseFont /Helvetica >>\nendobj\n"
    obj5 = f"5 0 obj\n<< /Length {len(stream_bytes)} >>\nstream\n".encode('ascii') + stream_bytes + b"\nendstream\nendobj\n"

    offsets = []
    current_pos = len(pdf_data)
    for obj in [obj1, obj2, obj3, obj4, obj5]:
        offsets.append(current_pos)
        pdf_data.extend(obj)
        current_pos = len(pdf_data)

    xref_pos = current_pos
    xref = f"xref\n0 6\n0000000000 65535 f \n".encode('ascii')
    for off in offsets:
        xref += f"{off:010d} 00000 n \n".encode('ascii')

    trailer = f"trailer\n<< /Size 6 /Root 1 0 R >>\nstartxref\n{xref_pos}\n%%EOF".encode('ascii')
    pdf_data.extend(xref)
    pdf_data.extend(trailer)

    with open(ruta, 'wb') as f:
        f.write(pdf_data)


def crear_archivos_demo(directorio=DIRECTORIO, cantidad=50, semilla=7):
    random.seed(semilla)
    os.makedirs(directorio, exist_ok=True)
    ahora = datetime.now()
    nombres_usados = set()
    creados = []

    # Datos.json: el archivo de inventario inicial
    with open(os.path.join(directorio, 'Datos.json'), 'w', encoding='utf-8') as f:
        json.dump(_registros(25), f, indent=2, ensure_ascii=False)

    while len(creados) < cantidad:
        ext = random.choice(['.json', '.csv', '.pdf', '.txt'])
        nombre = f'{random.choice(TEMAS)}_{random.choice(PERIODOS)}{ext}'
        if nombre in nombres_usados or nombre == 'Datos.json':
            continue
        nombres_usados.add(nombre)
        ruta = os.path.join(directorio, nombre)
        regs = _registros(random.choice([3, 8, 20, 60, 150, 400]))  # tamaños variados

        if ext == '.json':
            with open(ruta, 'w', encoding='utf-8') as f:
                json.dump(regs, f, indent=2, ensure_ascii=False)
        elif ext == '.csv':
            with open(ruta, 'w', newline='', encoding='utf-8') as f:
                w = csv.DictWriter(f, fieldnames=list(regs[0]))
                w.writeheader()
                w.writerows(regs)
        elif ext == '.txt':
            with open(ruta, 'w', encoding='utf-8') as f:
                f.write(f"REPORTE DE INVENTARIO - {nombre.upper()}\n" + "="*40 + "\n\n")
                for r in regs:
                    f.write(f"ID: {r['id']} | {r['producto']} | {r['categoria']} | Cant: {r['cantidad']} | ${r['precio']}\n")
        elif ext == '.pdf':
            _guardar_pdf(ruta, regs)

        # Fecha de modificación aleatoria en los últimos ~10 meses
        fecha = ahora - timedelta(days=random.randint(0, 300),
                                  hours=random.randint(0, 23),
                                  minutes=random.randint(0, 59))
        os.utime(ruta, (fecha.timestamp(), fecha.timestamp()))
        creados.append(nombre)

    print(f'Se crearon {len(creados)} archivos de ejemplo (+ Datos.json) incluyendo .json, .csv, .txt y .pdf en {directorio}')
    return creados


if __name__ == '__main__':
    crear_archivos_demo()

Se crearon 50 archivos de ejemplo (+ Datos.json) incluyendo .json, .csv, .txt y .pdf en /content/drive/My Drive/InventarioColab
